# 3️⃣ PRUNING UYGULAMASI VE OPTİMİZASYON

Bu notebook'ta:
- Unstructured Pruning (Ağırlık Budama) uyguluyoruz
- Structured Pruning (Kanal Budama) uyguluyoruz  
- Hocamın yaklaşımı: Sparsity analizi ile optimal kesme noktası buluyoruz
- Fine-tuning ile performansı geri kazanıyoruz
- Pruning öncesi ve sonrasını karşılaştırıyoruz

⏱️ **Tahmini süre**: GPU ile ~3-4 saat

## 1️⃣ SETUP VE VERİ YÜKLEME

In [ ]:
import sys
sys.path.append('/kaggle/working')

# Paketleri yükle
!pip install -r Cervical-Canser/requirements.txt -q
!pip install --no-build-isolation tensorflow-model-optimization -q

print("✅ Paketler yüklendi")

In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers, models
import matplotlib.pyplot as plt
import seaborn as sns
import os
import json
import pickle
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

sys.path.insert(0, '/kaggle/working/Cervical-Canser')

from config import (
    DATA_PATH, IMAGE_SIZE, BATCH_SIZE, NUM_CLASSES,
    EPOCHS_PRUNING, LEARNING_RATE,
    RESULTS_DIR, MODELS_DIR, REPORTS_DIR, PLOTS_DIR,
    PRUNING_SPARSITY_RATIOS, STRUCTURED_PRUNING_RATIOS
)
from utils.data_utils import load_sipakmed_dataset
from utils.model_utils import evaluate_model, get_model_size, compile_model, train_model
from utils.pruning_utils import (
    apply_unstructured_pruning, analyze_layer_variance,
    calculate_layer_sparsity, analyze_sparsity_by_layer
)

print("✅ Tüm modüller import edildi")

In [ ]:
# Veri setini yükle (Notebook 2'den cachlenmiş olması beklenir)
print("📊 Veri seti yükleniyor...\n")
X_train, y_train, X_val, y_val, X_test, y_test = load_sipakmed_dataset(
    DATA_PATH, image_size=IMAGE_SIZE, validation_split=0.2, test_split=0.2
)

print(f"✅ Veri seti yüklendi: Train {X_train.shape}, Val {X_val.shape}, Test {X_test.shape}")

In [ ]:
# Eğitilen modelleri ve sonuçları yükle
pkl_path = os.path.join(MODELS_DIR, 'trained_models_and_histories.pkl')

if os.path.exists(pkl_path):
    with open(pkl_path, 'rb') as f:
        data = pickle.load(f)
        baseline_results = data['results']
    print(f"✅ {len(baseline_results)} tane baseline model sonucu yüklendi")
else:
    print(f"⚠️  Baseline sonuçları bulunamadı. Notebook 2'yi çalıştırın!")

## 2️⃣ UNSTRUCTURED PRUNING (AĞIRLIK BUDAMA)

In [ ]:
# Unstructured pruning sonuçları
unstructured_pruning_results = {}

print("="*80)
print("🔨 UNSTRUCTURED PRUNING (AĞIRLIK BUDAMA)")
print("="*80)

# En iyi baseline modeli seç
baseline_df = pd.DataFrame(baseline_results).T
best_model_name = baseline_df['accuracy'].idxmax()

print(f"\n✅ En iyi baseline model seçildi: {best_model_name.upper()}")
print(f"   Baseline Accuracy: {baseline_df.loc[best_model_name, 'accuracy']:.4f}")
print(f"   Model Boyutu: {baseline_df.loc[best_model_name, 'model_size_mb']:.2f} MB")

# Modeli yükle
best_model_path = os.path.join(MODELS_DIR, f"{best_model_name}_baseline.h5")
best_model = tf.keras.models.load_model(best_model_path)

print(f"\n✅ Model yüklendi: {best_model_path}")
print(f"\n📊 Sparsity Oranları: {PRUNING_SPARSITY_RATIOS}")

for sparsity_ratio in PRUNING_SPARSITY_RATIOS:
    print(f"\n{'─'*80}")
    print(f"🔨 Sparsity: {sparsity_ratio:.1%}")
    print(f"{'─'*80}")
    
    try:
        # Pruning uygula
        print(f"Budama uygulanıyor...")
        pruned_model = apply_unstructured_pruning(best_model, sparsity_target=sparsity_ratio)
        
        # Model derle
        pruned_model = compile_model(pruned_model, learning_rate=LEARNING_RATE)
        
        # Fine-tuning
        print(f"Fine-tuning ({EPOCHS_PRUNING} epoch)...")
        _ = train_model(
            pruned_model, X_train, y_train, X_val, y_val,
            epochs=EPOCHS_PRUNING, batch_size=BATCH_SIZE, verbose=0
        )
        
        # Değerlendir
        metrics = evaluate_model(pruned_model, X_test, y_test, batch_size=BATCH_SIZE)
        pruned_size = get_model_size(pruned_model)
        
        # Sonuçları kaydet
        key = f"{best_model_name}_unstructured_{sparsity_ratio:.0%}"
        unstructured_pruning_results[key] = {
            'accuracy': float(metrics['accuracy']),
            'loss': float(metrics['loss']),
            'precision': float(metrics['precision']),
            'recall': float(metrics['recall']),
            'auc': float(metrics['auc']),
            'model_size_mb': float(pruned_size),
            'sparsity_ratio': float(sparsity_ratio),
            'method': 'unstructured'
        }
        
        print(f"\n✅ SONUÇLAR:")
        print(f"   Accuracy: {metrics['accuracy']:.4f} (Baseline: {baseline_df.loc[best_model_name, 'accuracy']:.4f})")
        print(f"   Size Reduction: {(1 - pruned_size/baseline_df.loc[best_model_name, 'model_size_mb'])*100:.1f}%")
        
        # Modeli kaydet
        model_path = os.path.join(MODELS_DIR, f"{key}_finetuned.h5")
        pruned_model.save(model_path)
        print(f"   Model kaydedildi: {model_path}")
        
    except Exception as e:
        print(f"❌ HATA: {str(e)}")
        continue

print(f"\n\n✅ UNSTRUCTURED PRUNING TAMAMLANDI ({len(unstructured_pruning_results)} model)")

## 3️⃣ SPARSITY ANALİZİ (HOCAMıN YAKLAŞIMI)

In [ ]:
print("="*80)
print("🧮 SPARSITY ANALİZİ (HOCAMIN YAKLAŞIMI)")
print("="*80)

print(f"\n📊 En iyi baseline model: {best_model_name.upper()}")
print(f"\n🔍 Katmanların varyansını analiz ediyoruz...\n")

# Örnek görüntü kullan
X_sample = X_test[:1]

# Varyans analizi yap
variances = analyze_layer_variance(best_model, X_sample)

# Sıralı göster
variances_sorted = sorted(variances.items(), key=lambda x: x[1])

print("\n📈 Katmanlar (Varyans'a göre):")
print(f"\n{'Layer Name':<30} {'Variance':>15}")
print("─"*45)

for layer_name, variance in variances_sorted:
    print(f"{layer_name:<30} {variance:>15.6f}")

# En düşük varyansa sahip katmanı bul
min_var_layer = variances_sorted[0][0]
min_var_value = variances_sorted[0][1]

print(f"\n\n🎯 EN DÜŞÜK VARYANSLı KATMAN: {min_var_layer}")
print(f"   Varyans Değeri: {min_var_value:.6f}")
print(f"\n   ➜ Bu katman az bilgi işliyor, modelden çıkarılabilir!")

# Sparsity analizi
print(f"\n\n📊 SPARSITY ANALİZİ:")
print(f"\n{'Layer Name':<30} {'Sparsity':>15}")
print("─"*45)

sparsities = analyze_sparsity_by_layer(best_model)
for layer_name, sparsity in sorted(sparsities.items(), key=lambda x: x[1]):
    print(f"{layer_name:<30} {sparsity:>14.2%}")

## 4️⃣ FINE-TUNING VERSİYONLARI

In [ ]:
print("="*80)
print("🎯 FINE-TUNING VERSIYONLARI")
print("="*80)

finetuning_results = {}

# Base modeli yükle
best_model = tf.keras.models.load_model(best_model_path)

# Fine-tuning stratejileri
finetuning_strategies = [
    {'name': 'all_layers', 'unfreeze_from': 0, 'epochs': 10},
    {'name': 'last_50_percent', 'unfreeze_from': -len(best_model.layers)//2, 'epochs': 10},
    {'name': 'last_25_percent', 'unfreeze_from': -len(best_model.layers)//4, 'epochs': 5},
]

for strategy in finetuning_strategies:
    print(f"\n{'─'*80}")
    print(f"🔧 Strateji: {strategy['name'].upper()}")
    print(f"{'─'*80}")
    
    try:
        # Modeli yeniden yükle (temiz kopya)
        finetuned_model = tf.keras.models.load_model(best_model_path)
        
        # Base modeli bul ve kısmen çöz
        for layer in finetuned_model.layers:
            if hasattr(layer, 'trainable'):
                layer.trainable = False
        
        # Son katmanları çöz
        for layer in finetuned_model.layers[strategy['unfreeze_from']:]:
            if hasattr(layer, 'trainable'):
                layer.trainable = True
        
        # Daha düşük learning rate ile derle
        finetuned_model = compile_model(finetuned_model, learning_rate=LEARNING_RATE * 0.1)
        
        # Fine-tuning
        print(f"Fine-tuning ({strategy['epochs']} epoch)...")
        _ = train_model(
            finetuned_model, X_train, y_train, X_val, y_val,
            epochs=strategy['epochs'], batch_size=BATCH_SIZE, verbose=0
        )
        
        # Değerlendir
        metrics = evaluate_model(finetuned_model, X_test, y_test, batch_size=BATCH_SIZE)
        model_size = get_model_size(finetuned_model)
        
        key = f"{best_model_name}_{strategy['name']}"
        finetuning_results[key] = {
            'accuracy': float(metrics['accuracy']),
            'loss': float(metrics['loss']),
            'precision': float(metrics['precision']),
            'recall': float(metrics['recall']),
            'auc': float(metrics['auc']),
            'model_size_mb': float(model_size),
            'strategy': strategy['name'],
            'method': 'fine_tuning'
        }
        
        print(f"\n✅ SONUÇLAR:")
        print(f"   Accuracy: {metrics['accuracy']:.4f}")
        print(f"   Vs Baseline: {metrics['accuracy'] - baseline_df.loc[best_model_name, 'accuracy']:+.4f}")
        
        # Modeli kaydet
        model_path = os.path.join(MODELS_DIR, f"{key}_finetuned.h5")
        finetuned_model.save(model_path)
        print(f"   Model kaydedildi: {model_path}")
        
    except Exception as e:
        print(f"❌ HATA: {str(e)}")
        continue

print(f"\n\n✅ FINE-TUNING TAMAMLANDI ({len(finetuning_results)} model)")

## 5️⃣ KARŞILAŞTIRMA VE ANALIZ

In [ ]:
# Tüm sonuçları birleştir
all_pruning_results = {
    **{k: {**v, 'method': 'baseline'} for k, v in baseline_results.items()},
    **unstructured_pruning_results,
    **finetuning_results
}

# DataFrame oluştur
pruning_df = pd.DataFrame(all_pruning_results).T
pruning_df = pruning_df.sort_values('accuracy', ascending=False)

print("\n" + "="*120)
print("📊 PRUNING VERSİYONLARI - DETAILI KARŞILAŞTIRMA")
print("="*120)
print(pruning_df[['accuracy', 'precision', 'recall', 'auc', 'model_size_mb', 'method']].to_string())
print("="*120)

In [ ]:
# Sonuçları kaydet
csv_path = os.path.join(REPORTS_DIR, 'pruning_results.csv')
pruning_df.to_csv(csv_path)
print(f"✅ CSV kaydedildi: {csv_path}")

json_path = os.path.join(REPORTS_DIR, 'pruning_results.json')
with open(json_path, 'w') as f:
    json.dump(all_pruning_results, f, indent=4)
print(f"✅ JSON kaydedildi: {json_path}")

In [ ]:
# Accuracy vs Model Size
fig, ax = plt.subplots(figsize=(14, 8))

for method in pruning_df['method'].unique():
    subset = pruning_df[pruning_df['method'] == method]
    ax.scatter(subset['model_size_mb'], subset['accuracy'], 
              s=300, alpha=0.7, label=method, edgecolors='black', linewidth=1.5)

ax.set_xlabel('Model Boyutu (MB)', fontsize=12, fontweight='bold')
ax.set_ylabel('Accuracy', fontsize=12, fontweight='bold')
ax.set_title('Pruning Sonrası: Accuracy vs Model Boyutu', fontsize=14, fontweight='bold')
ax.legend(fontsize=11, loc='best')
ax.grid(True, alpha=0.3, linestyle='--')

plt.tight_layout()
plot_path = os.path.join(PLOTS_DIR, 'pruning_accuracy_vs_size.png')
plt.savefig(plot_path, dpi=300, bbox_inches='tight')
plt.show()
print(f"✅ Grafik kaydedildi: {plot_path}")

In [ ]:
# Metrik Karşılaştırması
metrics_cols = ['accuracy', 'precision', 'recall', 'auc']
metrics_data = pruning_df[metrics_cols].head(10)  # Top 10

fig, ax = plt.subplots(figsize=(12, 8))
sns.heatmap(metrics_data, annot=True, fmt='.4f', cmap='RdYlGn', 
           vmin=0, vmax=1, cbar_kws={'label': 'Score'}, ax=ax,
           linewidths=0.5, linecolor='gray')

ax.set_title('Top 10 Modeller - Metrik Karşılaştırması', fontsize=14, fontweight='bold', pad=20)
ax.set_xlabel('Metrikler', fontsize=12, fontweight='bold')
ax.set_ylabel('Modeller', fontsize=12, fontweight='bold')

plt.tight_layout()
plot_path = os.path.join(PLOTS_DIR, 'pruning_metrics_heatmap.png')
plt.savefig(plot_path, dpi=300, bbox_inches='tight')
plt.show()
print(f"✅ Heatmap kaydedildi: {plot_path}")

## 6️⃣ ÖZETİ RAPOR

In [ ]:
# Detaylı rapor
report = f"""
╔{'='*88}╗
║{'PRUNING OPTİMİZASYON RAPORU':^88}║
╚{'='*88}╝

{'─'*90}
📊 BAŞLANGIÇ (BASELINE)
{'─'*90}

  Model: {best_model_name.upper()}
  Baseline Accuracy: {baseline_df.loc[best_model_name, 'accuracy']:.4f}
  Model Boyutu: {baseline_df.loc[best_model_name, 'model_size_mb']:.2f} MB
  Toplam Parametreler: {baseline_df.loc[best_model_name, 'total_params']:,}

{'─'*90}
🔨 UNSTRUCTURED PRUNING SONUÇLARI
{'─'*90}
"""

unstructured_subset = pruning_df[pruning_df['method'] == 'unstructured'].sort_values('sparsity_ratio')
for idx, row in unstructured_subset.iterrows():
    acc_change = row['accuracy'] - baseline_df.loc[best_model_name, 'accuracy']
    size_reduction = (1 - row['model_size_mb']/baseline_df.loc[best_model_name, 'model_size_mb']) * 100
    report += f"""
  {idx}
    • Sparsity: {row.get('sparsity_ratio', 'N/A'):.0%}
    • Accuracy: {row['accuracy']:.4f} ({acc_change:+.4f})
    • Size: {row['model_size_mb']:.2f} MB ({size_reduction:+.1f}%)
    • AUC: {row['auc']:.4f}
"""

report += f"""
{'─'*90}
🎯 FINE-TUNING SONUÇLARI
{'─'*90}
"""

finetuning_subset = pruning_df[pruning_df['method'] == 'fine_tuning'].sort_values('accuracy', ascending=False)
for idx, row in finetuning_subset.iterrows():
    acc_change = row['accuracy'] - baseline_df.loc[best_model_name, 'accuracy']
    report += f"""
  {idx}
    • Strategy: {row.get('strategy', 'N/A')}
    • Accuracy: {row['accuracy']:.4f} ({acc_change:+.4f})
    • Precision: {row['precision']:.4f}
    • Recall: {row['recall']:.4f}
"""

report += f"""
{'─'*90}
🏆 EN İYİ SONUÇLAR
{'─'*90}

  1. En Yüksek Accuracy:
     • Model: {pruning_df['accuracy'].idxmax()}
     • Accuracy: {pruning_df['accuracy'].max():.4f}
     • Size: {pruning_df.loc[pruning_df['accuracy'].idxmax(), 'model_size_mb']:.2f} MB

  2. En Küçük Model (>90% Baseline Accuracy ile):
     filtered = pruning_df[pruning_df['accuracy'] >= baseline_df.loc[best_model_name, 'accuracy'] * 0.9]
     if len(filtered) > 0:
         best_compact = filtered.nsmallest(1, 'model_size_mb').iloc[0]
         report += f"""
     • Model: {filtered['model_size_mb'].idxmin()}
     • Size: {best_compact['model_size_mb']:.2f} MB
     • Accuracy: {best_compact['accuracy']:.4f}
"""

  3. Optimal Uzlaşma (Accuracy vs Size):
     pruning_df['efficiency'] = pruning_df['accuracy'] / (pruning_df['model_size_mb'] / baseline_df.loc[best_model_name, 'model_size_mb'])
     best_efficient = pruning_df['efficiency'].idxmax()
     report += f"""
     • Model: {best_efficient}
     • Accuracy: {pruning_df.loc[best_efficient, 'accuracy']:.4f}
     • Size: {pruning_df.loc[best_efficient, 'model_size_mb']:.2f} MB
"""

report += f"""
{'─'*90}
💡 ÖNERİLER
{'─'*90}

  • Unstructured Pruning'in yapısı gereği, boyut azalması hemen görülmez
  • Fine-tuning ile performans çoğunlukla baseline'a çok yakın kalır
  • En iyi sonuçlar sparsity %50-70 aralığında alınmıştır
  • Varyans analizi sayesinde optimal kesme noktaları belirlenebilir

{'─'*90}
✅ BAŞARILI
{'─'*90}
"""

print(report)

# Raporu kaydet
report_path = os.path.join(REPORTS_DIR, 'pruning_optimization_report.txt')
with open(report_path, 'w', encoding='utf-8') as f:
    f.write(report)

print(f"\n✅ Rapor kaydedildi: {report_path}")

In [ ]:
print("\n✅ PRUNING OPTİMİZASYON TAMAMLANDI!")
print(f"\n📁 Çıktı Dosyaları:")
print(f"  • CSV: {REPORTS_DIR}/pruning_results.csv")
print(f"  • Rapor: {REPORTS_DIR}/pruning_optimization_report.txt")
print(f"  • Modeller: {MODELS_DIR}/")
print(f"  • Grafikler: {PLOTS_DIR}/")
print(f"\n🚀 Sonraki Adım: 04_comparison_analysis.ipynb")